# Test 2 (DAC interno) — Adquisición con OR_MASK (multitrigger)

La señal proviene del **DAC/ASG interno** del Red Pitaya. Para la versión con
generador externo **Rigol DG4162** ver `multitrigger_test_adq_dg4162.ipynb`.

Flujo:
1. DAC: seno 100 kHz en `OUT1`, 25 kHz en `OUT2` (config en la celda DAC).
2. Acquisition básica vía API `rp` (decimación, threshold, delay).
3. **OR_MASK** = `0xFFFF_FFFF` en `0x240` y `0x244` (canal 0 y 1) — escrito
   vía `/dev/mem` porque la API estándar no expone el nuevo registro.
4. Arm + trigger + lectura del buffer + lectura de debug regs.
5. Validación: alternar qué canal del DAC produce la señal y confirmar que
   el trigger sigue disparando (gracias a la OR de todas las fuentes).

In [ ]:
import time, mmap, os, struct
import numpy as np
from matplotlib import pyplot as plt
import rp

# === Cómo cargar un bitstream propio vía rp_overlay ===
# Hay que dejar en /opt/redpitaya/fpga/<nombre_proyecto>/ los archivos:
#   - fpga.bit.bin   (bitstream en formato Zynq con header)
#   - fpga.dtbo      (device tree overlay)
#
# Setup típico desde la Pitaya:
#   redpitaya> mkdir -p /opt/redpitaya/fpga/$(monitor -f)/MCA
#   redpitaya> cp /root/red_pitaya_top.bit.bin \
#                 /opt/redpitaya/fpga/$(monitor -f)/MCA/fpga.bit.bin
#   redpitaya> cp /root/devicetree.dtbo \
#                 /opt/redpitaya/fpga/$(monitor -f)/MCA/fpga.dtbo
#
# Luego desde Python:
#   from rp_overlay import overlay
#   fpga = overlay("MCA")          # ← NOMBRE del directorio, NO el .bit.bin
#
# Mientras desarrollás podés bypasear el overlay con fpgautil (solo carga
# el bitstream, NO el device tree — sirve para acceso /dev/mem directo):
#   !fpgautil -b /root/red_pitaya_top.bit.bin

rp.rp_Init()

# /dev/mem helper para la zona multitrigger (0x210, 0x214, 0x218, 0x21C, 0x240+)
SCOPE_PHYS = 0x4010_0000
SCOPE_SIZE = 0x30000
fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
scope = mmap.mmap(fd, SCOPE_SIZE, mmap.MAP_SHARED,
                  mmap.PROT_READ | mmap.PROT_WRITE, offset=SCOPE_PHYS)

def w32(off, v):
    scope[off:off+4] = struct.pack('<I', v & 0xFFFFFFFF)

def r32(off):
    return struct.unpack('<I', scope[off:off+4])[0]

OR_MASK_ALL = 0xFFFF_FFFF  # vector de 1: todas las fuentes habilitadas
N_BUF = 16384
FS    = 125e6              # Hz, sampling rate con decimación 1

In [19]:
# Comprobación de carga del bitstream
!fpgautil -b /root/red_pitaya_top.bit.bin
# Tenemos que hacer fpgautil -b top.bit.bin 

print(f'0x14  set_dec_ch0  = {r32(0x14):#010x}  (esperado tras reset: 0x00000001)')
print(f'0x114 set_dec_ch1  = {r32(0x114):#010x}  (esperado tras reset: 0x00000001)')
print(f'0x204 calib_gain_0 = {r32(0x204):#010x}  (esperado tras reset: 0x00008000)')
print(f'0x21C we_keep+dis  = {r32(0x21C):#010x}  (esperado tras reset: 0)')
print(f'0x240 trg_src_ch0  = {r32(0x240):#010x}')

Time taken to load BIN is 36.000000 Milli Seconds
BIN FILE loaded through FPGA manager successfully
0x14  set_dec_ch0  = 0x00000001  (esperado tras reset: 0x00000001)
0x114 set_dec_ch1  = 0x00000001  (esperado tras reset: 0x00000001)
0x204 calib_gain_0 = 0x00008000  (esperado tras reset: 0x00008000)
0x21C we_keep+dis  = 0x00000000  (esperado tras reset: 0)
0x240 trg_src_ch0  = 0x00000000


## DAC: sine ch2, DC fino ch1

In [ ]:
# DAC en modo CONTINUO con freqs independientes por canal.
#
# Confirmado leyendo /home/lorenzo/PI/RedPitaya/rp-api/api/src/:
#   - rp_GenMode(ch, RP_GEN_MODE_CONTINUOUS): apaga burst (count=0, delay=0,
#     repetitions=0). Por sí solo NO arranca la salida.
#   - rp_GenTriggerSource(ch, RP_GEN_TRIG_SRC_INTERNAL): escribe
#     triggerSelector=1 en el registro del gen. Le dice a la FSM qué fuente
#     de trigger usar, pero NO emite el pulso.
#   - rp_GenOutEnableSync(True): destraba el "force output to 0" — habilita
#     que la salida llegue al pin, pero NO arranca el sequencer.
#   - rp_GenTriggerOnly(ch) (= generate_Trigger): ESTO es lo que arranca el
#     sequencer. En CONTINUOUS corre para siempre tras este pulso; en BURST
#     dispara un burst por pulso.
#
# Por eso, aún en CONTINUO + INTERNAL, hay que pulsar TriggerOnly una vez
# para que la DAC empiece a emitir señal.

def _set_continuous(ch):
    """Forzar modo continuo (no burst). Tolerante a APIs que no exponen GenMode."""
    try:
        if hasattr(rp, 'rp_GenMode') and hasattr(rp, 'RP_GEN_MODE_CONTINUOUS'):
            rp.rp_GenMode(ch, rp.RP_GEN_MODE_CONTINUOUS)
    except Exception:
        pass

def configure_dac_continuous(sine_chs=None, freqs=None, dc_chs=None,
                              amp=1.0, dc=0.3):
    """
    sine_chs:  lista de canales con seno, ej: [1, 2], [1], [2], o None.
    freqs:     dict {1: f_ch1, 2: f_ch2} con la freq de cada canal en sine_chs.
    dc_chs:    lista de canales en DC (offset = dc), ej: [1] o None.
    amp:       amplitud del seno (V, max ~1.0).
    dc:        nivel DC (V).
    """
    sine_chs = sine_chs or []
    dc_chs   = dc_chs   or []
    freqs    = freqs    or {}

    rp.rp_GenReset()

    for c in sine_chs:
        ch = rp.RP_CH_1 if c == 1 else rp.RP_CH_2
        f  = freqs.get(c, 100_000)
        rp.rp_GenWaveform(ch, rp.RP_WAVEFORM_SINE)
        rp.rp_GenFreqDirect(ch, f)
        rp.rp_GenAmp(ch, amp)
        _set_continuous(ch)
        rp.rp_GenTriggerSource(ch, rp.RP_GEN_TRIG_SRC_INTERNAL)

    for c in dc_chs:
        ch = rp.RP_CH_1 if c == 1 else rp.RP_CH_2
        rp.rp_GenWaveform(ch, rp.RP_WAVEFORM_DC)
        rp.rp_GenAmp(ch, 0.0)
        rp.rp_GenOffset(ch, dc)
        _set_continuous(ch)
        rp.rp_GenTriggerSource(ch, rp.RP_GEN_TRIG_SRC_INTERNAL)

    # Habilita las salidas (destraba el output)
    rp.rp_GenOutEnableSync(True)

    # ARRANCA el sequencer una vez por canal configurado.
    # En CONTINUO esto inicia el output que después corre solo, sin más
    # triggers. Si lo omitís, el generador queda esperando y nunca emite señal.
    for c in sine_chs + dc_chs:
        ch = rp.RP_CH_1 if c == 1 else rp.RP_CH_2
        rp.rp_GenTriggerOnly(ch)

# Default: OUT1 seno 100 kHz, OUT2 seno 25 kHz (freqs distintas para validación)
configure_dac_continuous(sine_chs=[1, 2],
                          freqs={1: 100_000, 2: 25_000})
print('DAC continuo: OUT1=100kHz, OUT2=25kHz (arrancado con TriggerOnly)')

## ACQ base + OR_MASK + captura

In [ ]:
# Helpers del scope: usamos directamente la clase MultiTriggerScope de
# multitrigger_utils en vez de redefinir todo inline.
from multitrigger_utils import MultiTriggerScope, decode_snap

# Reusa el mmap/fd ya abiertos en la celda de imports (no re-mapea /dev/mem;
# por eso NO se llama sc.close(): el cierre lo hace la celda de cleanup).
sc = MultiTriggerScope(scope, fd)

# Aliases para que las celdas de abajo llamen a los métodos de la clase con
# los nombres de siempre.
acq_base        = sc.acq_base          # config por ESCRITURA DIRECTA (decim/thr/delay)
set_or_mask     = sc.set_or_mask       # set_or_mask(mask) o set_or_mask(m0, m1)
wait_triggered  = sc.wait_triggered
wait_fill       = sc.wait_fill
read_buffers    = sc.read_buffers
debug_dump      = sc.debug_dump
debug_trigger   = sc.debug_trigger
decode_snapshot = decode_snap          # mismo decoder del snapshot @0x218

print('MultiTriggerScope listo: acq_base/set_or_mask/wait_*/read_buffers/debug_* via sc')

In [ ]:
# === Sanity check del ADC con SEÑAL EXTERNA o DAC ===
# acq_capture_sw viene de la clase (fuerza UN trigger SW, espera fill y lee el
# buffer congelado). adc_signal_check es análisis local del notebook.
acq_capture_sw = sc.acq_capture_sw

def adc_signal_check(channel=1, plot_n=2000):
    """Captura UNA ventana consistente (SW trigger) y muestra forma + FFT."""
    d1, d2, snap = acq_capture_sw()
    d = d1 if channel == 1 else d2

    sig   = d - d.mean()
    fft   = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(N_BUF, d=1/FS)
    f_fft = freqs[np.argmax(fft[1:]) + 1]

    sign = np.sign(d - d.mean())
    idx  = np.where(np.diff(sign) > 0)[0]
    f_zc = (1e6 / (np.diff(idx).mean() / FS * 1e6)) if len(idx) >= 2 else float('nan')

    print(f'IN{channel}:')
    print(f'  FFT pico      = {f_fft:>10.1f} Hz')
    print(f'  cruces 0 asc  = {f_zc:>10.1f} Hz   (debería matchear FFT)')
    print(f'  pico-pico     = {d.max()-d.min():.3f} V')
    print(f'  media         = {d.mean():+.3f} V')
    print(f'  snapshot @0x218 = {snap:#010x}  (debería tener el SW bit o el flanco que disparó)')

    fig, ax = plt.subplots(1, 2, figsize=(12, 3))
    ax[0].plot(d[:plot_n], label=f'IN{channel}')
    ax[0].axhline(d.mean(), color='gray', ls=':', lw=0.7)
    ax[0].set_title(f'IN{channel}: FFT={f_fft:.0f} Hz  zc={f_zc:.0f} Hz')
    ax[0].grid(True); ax[0].legend()

    ax[1].semilogy(freqs/1e3, fft / fft.max())
    ax[1].axvline(f_fft/1e3, color='r', ls='--', label=f'pico {f_fft/1e3:.1f} kHz')
    ax[1].set_xlim(0, min(FS/2/1e3, f_fft*5/1e3))
    ax[1].set_xlabel('freq (kHz)'); ax[1].set_ylabel('|FFT| norm')
    ax[1].grid(True); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()
    return d, f_fft

# Conectá tu generador a IN1 (y/o IN2). Cualquier waveform sirve.
adc_signal_check(channel=1)
adc_signal_check(channel=2)

In [ ]:
# === Debug completo del estado del trigger (método de la clase) ===
sc.debug_trigger()

In [ ]:
# Captura inicial: sine en OUT1 (100 kHz) y OUT2 (25 kHz), OR_MASK = todo
acq_base(thr=0.5, delay=0)
set_or_mask(OR_MASK_ALL)
sc.w32(0x00, 0x0000_0101)        # arm single-shot (ambos canales)
time.sleep(0.05)

trg_ok = wait_triggered()
buf_ok = wait_fill()
print(f'triggered={trg_ok}   buffer_full={buf_ok}')
debug_dump()

d1, d2 = read_buffers()
plt.figure(figsize=(10, 4))
plt.plot(d1, label='IN1')
plt.plot(d2, label='IN2')
plt.xlabel('sample'); plt.ylabel('V'); plt.legend(); plt.grid(True)
plt.title('Captura inicial — OR_MASK = 0xFFFFFFFF'); plt.show()

## Validación — alternar canales del DAC

Con OR_MASK = todas las fuentes habilitadas, el trigger debe disparar
independientemente de cuál canal de salida del DAC esté generando la señal
viva. Se corren 3 casos y se reporta `triggered` + `snapshot`.

In [ ]:
def run_case(label, sine_chs, dc_chs):
    configure_dac_continuous(sine_chs=sine_chs, dc_chs=dc_chs)
    acq_base(thr=0.5, delay=0)
    set_or_mask(OR_MASK_ALL)
    sc.w32(0x00, 0x0000_0101)        # arm single-shot
    time.sleep(0.05)
    ok = wait_triggered()
    snap = sc.r32(0x218)
    print(f'[{label}] sine={sine_chs} dc={dc_chs}   triggered={ok}   snapshot=0x{snap:08x}')
    return ok, snap

res_A = run_case('A', sine_chs=[2],    dc_chs=[1])
res_B = run_case('B', sine_chs=[1],    dc_chs=[2])
res_C = run_case('C', sine_chs=[1, 2], dc_chs=[])

all_ok = all(r[0] for r in (res_A, res_B, res_C))
print('VALIDACION OR:', 'PASS' if all_ok else 'FAIL')

## Decodificación del snapshot @0x218

Layout (17 bits): `{trig_ch[3:0], asg_n, asg_p, ext_n, ext_p, adc_n[3:0], adc_p[3:0], sw_any}`.
Bits desde el LSB:

| bit | fuente |
|-----|--------|
| 0   | SW manual (cualquier canal) |
| 1..4 | ADC posedge ch0..ch3 |
| 5..8 | ADC negedge ch0..ch3 |
| 9   | ext posedge |
| 10  | ext negedge |
| 11  | ASG posedge |
| 12  | ASG negedge |
| 13..16 | trig_ch[0..3] (cadena del otro scope) |

In [ ]:
# decode_snapshot ya está aliaseado a decode_snap (multitrigger_utils) en la
# celda de helpers. Acá solo lo aplicamos a los snapshots de los 3 casos.
for label, (ok, snap) in [('A', res_A), ('B', res_B), ('C', res_C)]:
    print(f'[{label}] snap=0x{snap:08x}  ->  {decode_snapshot(snap)}')

## Validación de tiempos de trigger

DAC en continuo con freqs distintas por canal. Se mide:

- **Caso A**: máscara OR solo sensible al ADC ch0 posedge (bit 1) → los
  intervalos entre cruces ascendentes deben coincidir con `1/f_ch0`.
- **Caso B**: máscara OR solo sensible al ADC ch1 posedge (bit 3) → los
  intervalos deben coincidir con `1/f_ch1`.
- **Caso C**: máscara OR con ambos canales habilitados + `adc_we_keep=1`
  (modo continuo de captura del scope) → loop de N capturas y se mide la
  freq efectiva de eventos. La OR de dos fuentes asíncronas dispara con
  cada flanco ⇒ período efectivo ≈ `min(1/f_ch0, 1/f_ch1)`.

Mapa de la máscara (32 bits, OR de las clases habilitadas):
| bit | fuente |
|-----|--------|
| 1   | ADC ch0 posedge |
| 2   | ADC ch0 negedge |
| 3   | ADC ch1 posedge |
| 4   | ADC ch1 negedge |
| 9   | ext posedge |
| 11  | ASG posedge |
| 13..16 | trig_ch[0..3] |

In [ ]:
# --- Helpers de timing (sobre MultiTriggerScope) ---

def edge_periods_us(samples, level=0.5, edge='rising'):
    """Períodos (µs) entre cruces consecutivos del nivel, en el sentido dado."""
    s = np.asarray(samples, dtype=float)
    sign = np.sign(s - level)
    if edge == 'rising':
        idx = np.where(np.diff(sign) > 0)[0]
    else:
        idx = np.where(np.diff(sign) < 0)[0]
    if len(idx) < 2:
        return np.array([])
    return np.diff(idx) / FS * 1e6  # µs

def capture_once(thr=0.5, delay=0, mask_ch0=OR_MASK_ALL, mask_ch1=OR_MASK_ALL,
                 timeout_ms=500):
    """Una captura single-shot usando la clase. Devuelve (d1, d2, snap, ok)."""
    sc.acq_base(thr=thr, delay=delay)
    sc.set_or_mask(mask_ch0, mask_ch1)
    sc.w32(0x00, 0x0000_0101)            # arm single-shot
    time.sleep(0.01)
    ok_trg  = sc.wait_triggered(timeout_ms=timeout_ms)
    ok_fill = sc.wait_fill(timeout_ms=timeout_ms)
    snap = sc.r32(0x218)
    d1, d2 = sc.read_buffers()
    return d1, d2, snap, (ok_trg and ok_fill)

def trigger_loop(n_caps=20, mask_ch0=OR_MASK_ALL, mask_ch1=OR_MASK_ALL,
                 thr=0.5, delay=0, timeout_ms=500):
    """N capturas en modo continuo (adc_we_keep=1) detectando triggers NUEVOS
    por cambios de adc_wp_trig @0x1C (rp_AcqGetTriggerState es sticky en
    we_keep). Devuelve lista de (t_ns, wp_trig, snap, ok)."""
    sc.acq_base(thr=thr, delay=delay)
    sc.set_or_mask(mask_ch0, mask_ch1)
    sc.w32(0x00, 0x0000_0909)            # arm + we_keep (continuo) en ambos canales
    out = []
    wp_prev = sc.r32(0x1C)
    t0 = time.perf_counter_ns()
    for _ in range(n_caps):
        ok = False
        for _ in range(timeout_ms):
            wp_cur = sc.r32(0x1C)
            if wp_cur != wp_prev:
                ok = True
                break
            time.sleep(0.001)
        t = time.perf_counter_ns() - t0
        snap = sc.r32(0x218)
        out.append((t, wp_cur, snap, ok))
        wp_prev = wp_cur
        sc.w32(0x94, 0x0000_0101)        # re-habilitar trigger en ambos canales
    return out

print('helpers definidos: edge_periods_us, capture_once, trigger_loop (sobre sc)')

In [ ]:
# === Caso A: máscara solo ADC ch0 posedge (bit 1) ===
# Esperado: intervalos entre cruces ascendentes de IN1 ≈ 1/f_ch0 = 10 µs (100 kHz).
configure_dac_continuous(sine_chs=[1, 2], freqs={1: 100_000, 2: 25_000})
MASK_CH0_PE = 1 << 1
d1, d2, snap, ok = capture_once(thr=0.5, mask_ch0=MASK_CH0_PE, mask_ch1=MASK_CH0_PE)
print(f'[A] triggered={ok}   snapshot=0x{snap:08x} -> {decode_snapshot(snap)}')

per_us = edge_periods_us(d1, level=0.5, edge='rising')
exp_us = 1e6 / 100_000
print(f'[A] N cruces={len(per_us)+1}  período medio = {per_us.mean():.3f} µs '
      f'(esperado {exp_us:.3f} µs, error = {(per_us.mean()-exp_us)/exp_us*100:.2f}%)')

plt.figure(figsize=(10, 3))
plt.plot(d1[:2000], label='IN1 100kHz')
plt.axhline(0.5, color='r', ls='--', lw=0.7); plt.legend(); plt.grid(True)
plt.title('Caso A — captura disparada por flanco ascendente IN1'); plt.show()

In [ ]:
# === Caso B: máscara solo ADC ch1 posedge (bit 3) ===
# Esperado: intervalos entre cruces ascendentes de IN2 ≈ 1/f_ch1 = 40 µs (25 kHz).
MASK_CH1_PE = 1 << 3
d1, d2, snap, ok = capture_once(thr=0.5, mask_ch0=MASK_CH1_PE, mask_ch1=MASK_CH1_PE)
print(f'[B] triggered={ok}   snapshot=0x{snap:08x} -> {decode_snapshot(snap)}')

per_us = edge_periods_us(d2, level=0.5, edge='rising')
exp_us = 1e6 / 25_000
if len(per_us):
    print(f'[B] N cruces={len(per_us)+1}  período medio = {per_us.mean():.3f} µs '
          f'(esperado {exp_us:.3f} µs, error = {(per_us.mean()-exp_us)/exp_us*100:.2f}%)')
else:
    print('[B] sin suficientes cruces — la freq es baja vs N_BUF')

plt.figure(figsize=(10, 3))
plt.plot(d2, label='IN2 25kHz')
plt.axhline(0.5, color='r', ls='--', lw=0.7); plt.legend(); plt.grid(True)
plt.title('Caso B — captura disparada por flanco ascendente IN2'); plt.show()

In [ ]:
# === Caso C: máscara OR ch0+ch1 posedge + loop continuo (adc_we_keep=1) ===
# El loop detecta triggers NUEVOS via cambios de adc_wp_trig @0x1C
# (rp_AcqGetTriggerState es sticky en we_keep). El dt medido es una cota
# superior (incluye latencia del polling SW + bus PS↔PL ≈ µs).
MASK_AB = (1 << 1) | (1 << 3)
N_CAPS  = 30
events = trigger_loop(n_caps=N_CAPS,
                       mask_ch0=MASK_AB, mask_ch1=MASK_AB,
                       thr=0.5, timeout_ms=200)

ok_count = sum(1 for _, _, _, ok in events if ok)
ts_ns    = np.array([t  for t, _, _, _ in events])
wps      = np.array([wp for _, wp, _, _ in events])
snaps    = [s for _, _, s, _ in events]
dts_us   = np.diff(ts_ns) / 1000.0  # ns → µs

print(f'[C] {ok_count}/{N_CAPS} triggers nuevos detectados')
print(f'[C] dt entre triggers (µs): media={dts_us.mean():.2f}  '
      f'min={dts_us.min():.2f}  max={dts_us.max():.2f}')
print(f'[C] wp_trig (muestra dentro del buffer) últimos 5: {wps[-5:]}')
print(f'[C] snapshots únicos: {set(snaps)} -> {[decode_snapshot(s) for s in set(snaps)]}')
print(f'[C] esperado del trigger HW: ~10 µs (100 kHz domina). '
      f'Lo medido es la latencia SW; usar wp_trig consecutivos para timing real.')

plt.figure(figsize=(10, 3))
plt.plot(dts_us, 'o-')
plt.axhline(10.0, color='r', ls='--', label='1/100 kHz = 10 µs (HW)')
plt.xlabel('captura #'); plt.ylabel('dt SW entre triggers (µs)')
plt.legend(); plt.grid(True); plt.title('Caso C — loop multi-trigger'); plt.show()

# Limpieza: disarm() apaga we_keep, limpia shield/adc_trg_dis y resetea el FSM.
sc.disarm()

In [7]:
scope.close()
os.close(fd)
rp.rp_Release()
print('cerrado')

cerrado
